# TQ-IoT: Tabular Multi-Objective RL — Real FIT IoT-Lab Data
**Authors:** Abhay Singh & Peeyush Kelkar  
**Stockholm University — Spring 2026**

---

This notebook replaces the original synthetic sensor generation with **real FIT IoT-Lab sensor data**.  
Everything else — Q-tables, Chebyshev scalarisation, AoI, battery, training, evaluation, Pareto plots — stays exactly the same.

### Folder structure expected
```
project/
├── this_notebook.ipynb
├── data/
│   └── raw/
│       └── exp_436032/
│           ├── exp_436032_sensor_stream_raw.txt
│           ├── consumption/
│               ├── m3_10.oml
│               ├── m3_11.oml
│               ├── m3_12.oml
│               ├── m3_13.oml
│               └── m3_14.oml
└── results/
```

### Actions
| Action | Cost | AoI effect |
|--------|------|------------|
| 0 — sleep | 1 unit | AoI grows |
| 1 — wait (sense only) | 5 units | AoI grows |
| 2 — transmit | 10 units | AoI resets to 1 |


## 1. Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Tuple, List, Dict, Optional

os.makedirs("data/processed", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Libraries loaded.")


## 2. Load and Parse FIT IoT-Lab Sensor Data

**Source:** `data/raw/exp_436032/exp_436032_sensor_stream_raw.txt`

Raw line format:
```
timestamp ; node_id ; [cmd >] measurement_type: value unit
```
Examples:
```
1776012342.562994;m3-10;Chip temperature measure: 3.01E1
1776012342.563410;m3-10;cmd > Luminosity measure: 4.8828125E-1 lux
1776012342.563615;m3-10;cmd > Pressure measure: 9.983738E2 mabar
```

We parse all three sensors and build one clean row per (node, timestamp).


In [ ]:
# ── FIT IoT-Lab sensor file path ──────────────────────────────────────────
SENSOR_FILE = "data/raw/exp_436032/exp_436032_sensor_stream_raw.txt"

def parse_sensor_stream(path: str) -> pd.DataFrame:
    """
    Parse the raw FIT IoT-Lab serial log.

    Returns a DataFrame with one row per sensor reading:
        timestamp, node_id, sensor_type, value
    """
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(';')
            if len(parts) < 3:
                continue
            try:
                ts   = float(parts[0])
                node = parts[1].strip()
                raw  = parts[2].strip().replace('cmd >', '').strip()
            except ValueError:
                continue
            if not raw:
                continue

            # ── identify sensor type and extract numeric value ──────────────
            if 'Chip temperature' in raw:
                stype = 'temperature'
                val   = raw.split(':')[1].strip()
            elif 'Luminosity' in raw:
                stype = 'luminosity'
                val   = raw.split(':')[1].strip().split('lux')[0]
            elif 'Pressure' in raw:
                stype = 'pressure'
                val   = raw.split(':')[1].strip().split('mabar')[0]
            else:
                continue

            try:
                # convert scientific notation e.g. 3.01E1 → 30.1
                records.append({
                    'timestamp': ts,
                    'node_id'  : node,
                    'sensor_type': stype,
                    'value'    : float(val),
                })
            except ValueError:
                continue

    df = pd.DataFrame(records)
    df = df.sort_values(['timestamp', 'node_id']).reset_index(drop=True)
    return df


# ── parse ──────────────────────────────────────────────────────────────────
raw_df = parse_sensor_stream(SENSOR_FILE)
print(f"Total sensor readings parsed : {len(raw_df):,}")
print(f"Nodes found                  : {sorted(raw_df['node_id'].unique())}")
print(f"Sensor types                 : {raw_df['sensor_type'].unique().tolist()}")
print(f"Time span                    : {raw_df['timestamp'].max() - raw_df['timestamp'].min():.0f} seconds")
raw_df.head(9)


## 3. Build Clean Per-Step DataFrame

We pivot the long-format sensor readings into one row per (node, step) with columns for each sensor.  
Each step corresponds to one ~5-second sampling interval — the natural cadence of the FIT IoT-Lab nodes.

Saved to: `data/processed/fit_iot_clean.csv`


In [ ]:
def build_clean_dataframe(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot raw sensor readings into one row per (node_id, step).

    Columns: timestamp, node_id, temperature, luminosity, pressure
    Sorted by timestamp and node_id.
    """
    nodes = sorted(raw_df['node_id'].unique())
    all_rows = []

    for node in nodes:
        ndf = raw_df[raw_df['node_id'] == node]

        # get temperature timestamps as step anchors (~5 s cadence)
        temps = (ndf[ndf['sensor_type'] == 'temperature']
                 .sort_values('timestamp').reset_index(drop=True))

        for i, row in temps.iterrows():
            ts = row['timestamp']

            # find the closest luminosity and pressure within ±1 second
            def closest(stype):
                sub = ndf[(ndf['sensor_type'] == stype) &
                          (ndf['timestamp'] >= ts - 1.0) &
                          (ndf['timestamp'] <= ts + 1.0)]
                if len(sub) == 0:
                    return np.nan
                idx = (sub['timestamp'] - ts).abs().idxmin()
                return sub.loc[idx, 'value']

            all_rows.append({
                'timestamp'  : ts,
                'node_id'    : node,
                'temperature': round(row['value'], 4),
                'luminosity' : round(closest('luminosity'), 4),
                'pressure'   : round(closest('pressure'), 4),
            })

    clean = (pd.DataFrame(all_rows)
             .sort_values(['timestamp', 'node_id'])
             .reset_index(drop=True))
    return clean


# ── FIT IoT-Lab data: this is the real sensor stream ─────────────────────────
clean_df = build_clean_dataframe(raw_df)

# save processed data
clean_df.to_csv("data/processed/fit_iot_clean.csv", index=False)
print(f"Clean dataframe: {len(clean_df)} rows  ({clean_df['node_id'].nunique()} nodes)")
print(f"Saved to: data/processed/fit_iot_clean.csv")
clean_df.head(10)


## 4. FITIoTDataFeed — Real Sensor Feed for the RL Environment

This class replaces the random sensor generation in the original notebook.  
At every environment step it returns the next real sensor reading from the dataset.

**Key rule:** The data stream advances at every step regardless of the action taken —  
even when the agent sleeps, time moves forward through the real recorded data.  
AoI only resets when the agent transmits successfully.


In [ ]:
class FITIoTDataFeed:
    """
    Provides real FIT IoT-Lab sensor values to the RL environment
    step by step, cycling through the recorded data.

    Replaces the original:
        self.sensed_value = self.np_rng.normal(...)

    Usage:
        feed = FITIoTDataFeed(clean_df, node_id='m3-10')
        feed.reset()
        temp, lux, pressure = feed.next()   # call once per env step
    """

    def __init__(self, df: pd.DataFrame, node_id: str = 'm3-10'):
        node_df = df[df['node_id'] == node_id].reset_index(drop=True)
        self.timestamps   = node_df['timestamp'].values
        self.temperatures = node_df['temperature'].values
        self.luminosities = node_df['luminosity'].values
        self.pressures    = node_df['pressure'].values
        self.n            = len(node_df)
        self.node_id      = node_id
        self.idx          = 0

    def reset(self):
        """Reset to beginning of the sensor stream."""
        self.idx = 0

    def next(self) -> Tuple[float, float, float]:
        """
        Return the next real sensor reading (temperature, luminosity, pressure).
        Cycles back to start when the stream is exhausted.
        """
        i = self.idx % self.n
        vals = (
            float(self.temperatures[i]),
            float(self.luminosities[i]),
            float(self.pressures[i]),
        )
        self.idx += 1
        return vals

    def peek_prev(self) -> Tuple[float, float, float]:
        """Return the most recently consumed reading (for delta_v calculation)."""
        i = max(0, (self.idx - 1)) % self.n
        return (
            float(self.temperatures[i]),
            float(self.luminosities[i]),
            float(self.pressures[i]),
        )

    def __len__(self):
        return self.n


# ── quick sanity check ────────────────────────────────────────────────────────
feed_test = FITIoTDataFeed(clean_df, node_id='m3-10')
feed_test.reset()
print("First 5 real sensor readings for m3-10:")
print(f"{'Step':<6} {'Temp (°C)':<12} {'Lux':<12} {'Pressure (mbar)'}")
for i in range(5):
    t, l, p = feed_test.next()
    print(f"{i:<6} {t:<12.4f} {l:<12.4f} {p:.4f}")


## 5. Environment Configuration

In [ ]:
@dataclass
class EnvConfig:
    battery_cap       : int   = 100
    energy_cost_sleep : int   = 1      # sleep   → 1 unit
    energy_cost_sense : int   = 5      # wait    → 5 units
    energy_cost_tx    : int   = 10     # transmit→ 10 units
    tx_success_prob   : float = 0.90   # prob of successful transmission
    aoi_cap           : int   = 20     # maximum AoI value
    horizon           : int   = 200    # max steps per episode
    channel_trans     : np.ndarray = field(default_factory=lambda: np.array([
        [0.80, 0.15, 0.05],
        [0.10, 0.75, 0.15],
        [0.05, 0.20, 0.75],
    ], dtype=float))

print("EnvConfig defined.")


## 6. TQIoTEnv — Environment Using Real Sensor Data

The key change from the original notebook:

| Original | Rewritten |
|---|---|
| `self.sensed_value = self.np_rng.normal(...)` | `self.temperature, self.luminosity, self.pressure = self.feed.next()` |

Everything else — battery, AoI, channel, actions, rewards — is unchanged.


In [ ]:
class TQIoTEnv:
    """
    IoT scheduling environment driven by real FIT IoT-Lab sensor data.

    State: (battery_bin, aoi_bin, delta_v_bin, channel_bin)
      battery_bin : 0-5   how full the battery is
      aoi_bin     : 0-4   how stale the data is
      delta_v_bin : 0-2   how much the sensor value changed
      channel_bin : 0-2   wireless channel quality (poor/medium/good)

    Actions:
      0 = sleep     cost=1   AoI grows
      1 = wait      cost=5   AoI grows   (sense but don't transmit)
      2 = transmit  cost=10  AoI resets if successful

    Rewards:
      r_E = -energy_cost / 10          (energy reward)
      r_F = -aoi / aoi_cap             (freshness reward)

    FIT IoT-Lab data:
      self.feed.next() is called at EVERY step regardless of action,
      so the real data stream always advances with time.
      AoI only resets on successful transmit.
    """

    ACTIONS = {0: 'sleep', 1: 'wait', 2: 'transmit'}

    def __init__(self, cfg: EnvConfig, feed: FITIoTDataFeed, seed: int = 0):
        self.cfg  = cfg
        self.feed = feed
        self.rng  = np.random.default_rng(seed)

    def _battery_bin(self, b):
        if b > 80: return 5
        elif b > 60: return 4
        elif b > 40: return 3
        elif b > 20: return 2
        elif b > 5:  return 1
        else: return 0

    def _aoi_bin(self, a):
        if a <= 1:    return 0
        elif a <= 4:  return 1
        elif a <= 8:  return 2
        elif a <= 12: return 3
        else:         return 4

    def _delta_v_bin(self, delta: float) -> int:
        """
        Bin the change in temperature since last step.
        Uses REAL sensor delta — no random generation.
        """
        if delta < 0.2:   return 0   # small change
        elif delta < 0.5: return 1   # medium change
        else:             return 2   # large change

    def _obs(self):
        bb  = self._battery_bin(self.battery)
        ab  = self._aoi_bin(self.aoi)
        dvb = self._delta_v_bin(self.delta_v)
        return (bb, ab, dvb, self.channel)

    def reset(self):
        self.battery = self.cfg.battery_cap
        self.aoi     = 1
        self.channel = int(self.rng.choice([0, 1, 2], p=[0.3, 0.4, 0.3]))
        self.t       = 0

        # ── FIT IoT-Lab: reset the real data feed ─────────────────────────
        self.feed.reset()
        self.temperature, self.luminosity, self.pressure = self.feed.next()
        self.prev_temperature = self.temperature
        self.delta_v = 0.0

        return self._obs()

    def step(self, action: int):
        assert action in (0, 1, 2)
        cfg = self.cfg

        # ── FIT IoT-Lab: advance through the REAL sensor stream ───────────
        # This happens every step regardless of action (sleep, wait, transmit)
        self.prev_temperature = self.temperature
        self.temperature, self.luminosity, self.pressure = self.feed.next()
        self.delta_v = abs(self.temperature - self.prev_temperature)

        # ── energy cost ───────────────────────────────────────────────────
        cost = {0: cfg.energy_cost_sleep,
                1: cfg.energy_cost_sense,
                2: cfg.energy_cost_tx}[action]

        if self.battery < cost:
            return self._obs(), (-float(self.battery)/10, -self.aoi/cfg.aoi_cap), True, {}

        self.battery -= cost

        # ── AoI update ────────────────────────────────────────────────────
        # AoI resets ONLY on successful transmit
        if action == 2 and self.channel >= 1:
            tx_success = self.rng.random() < cfg.tx_success_prob
            if tx_success:
                self.aoi = 1
            else:
                self.aoi = min(self.aoi + 1, cfg.aoi_cap)
        else:
            self.aoi = min(self.aoi + 1, cfg.aoi_cap)

        # ── channel evolves via Markov chain ──────────────────────────────
        self.channel = int(np.argmax(
            self.rng.multinomial(1, cfg.channel_trans[self.channel])))

        # ── rewards ───────────────────────────────────────────────────────
        r_E = -cost / 10.0                    # energy reward
        r_F = -self.aoi / cfg.aoi_cap         # freshness reward

        self.t += 1
        done = (self.battery <= 0 or self.t >= cfg.horizon)
        return self._obs(), (r_E, r_F), done, {
            'temperature': self.temperature,
            'luminosity' : self.luminosity,
            'pressure'   : self.pressure,
            'delta_v'    : self.delta_v,
        }


print("TQIoTEnv defined (using real FIT IoT-Lab data).")


## 7. Agent — Two Q-Tables + Chebyshev Scalarisation

In [ ]:
@dataclass
class AgentConfig:
    gamma                 : float = 0.95
    alpha                 : float = 0.2
    epsilon_start         : float = 0.25
    epsilon_end           : float = 0.05
    epsilon_decay_episodes: int   = 300
    weights: List[Tuple[float,float]] = field(
        default_factory=lambda: [(round(w,1), round(1-w,1))
                                 for w in np.linspace(0.0, 1.0, 11)])

N_BATTERY  = 6
N_AOI      = 5
N_DELTA_V  = 3
N_CHANNEL  = 3
N_ACTIONS  = 3
N_STATES   = N_BATTERY * N_AOI * N_DELTA_V * N_CHANNEL   # 270

def state_to_index(s: Tuple[int,int,int,int]) -> int:
    bb, ab, dvb, ch = s
    return ((bb * N_AOI + ab) * N_DELTA_V + dvb) * N_CHANNEL + ch


class TabularMORLAgent:
    """
    Two Q-tables: QE (energy), QF (freshness).
    Chebyshev scalarisation combines them at decision time.

    score(a) = -max( wE*|QE(s,a)-z*E|, wF*|QF(s,a)-z*F| )
    Pick action with highest score.
    """

    def __init__(self, cfg: AgentConfig = AgentConfig(), seed: int = 0):
        self.cfg   = cfg
        self.rng   = random.Random(seed)
        self.QE    = np.zeros((N_STATES, N_ACTIONS))
        self.QF    = np.zeros((N_STATES, N_ACTIONS))
        self.ep    = 0

    def epsilon(self):
        e0, e1, N = self.cfg.epsilon_start, self.cfg.epsilon_end, self.cfg.epsilon_decay_episodes
        return e0 + (e1 - e0) * min(self.ep, N) / N

    def chebyshev_scores(self, s: int, w: Tuple[float,float]) -> np.ndarray:
        wE, wF  = w
        zE, zF  = np.max(self.QE[s]), np.max(self.QF[s])
        return -np.maximum(wE * (zE - self.QE[s]), wF * (zF - self.QF[s]))

    def select_action(self, s: int, w: Tuple[float,float], explore: bool = True) -> int:
        if explore and self.rng.random() < self.epsilon():
            return self.rng.randrange(N_ACTIONS)
        scores = self.chebyshev_scores(s, w)
        return int(np.argmax(scores))

    def update(self, s, a, rE, rF, s_next):
        g = self.cfg.gamma
        lr = self.cfg.alpha
        self.QE[s, a] += lr * (rE + g * np.max(self.QE[s_next]) - self.QE[s, a])
        self.QF[s, a] += lr * (rF + g * np.max(self.QF[s_next]) - self.QF[s, a])


print(f"Agent defined. State space: {N_STATES}, Actions: {N_ACTIONS}")
print(f"Weight configurations: {AgentConfig().weights}")


## 8. Training — All 11 Weight Configurations

In [ ]:
def train(clean_df: pd.DataFrame,
          agent_cfg: AgentConfig = AgentConfig(),
          env_cfg: EnvConfig = EnvConfig(),
          train_episodes_per_weight: int = 100,
          seed: int = 42) -> Tuple[TabularMORLAgent, List, Dict]:
    """
    Train TQ-IoT on real FIT IoT-Lab data.
    All 11 weight configurations are interleaved each epoch.
    One episode = one full pass through a node's sensor stream.
    """
    random.seed(seed)
    np.random.seed(seed)

    agent          = TabularMORLAgent(agent_cfg, seed=seed)
    nodes          = sorted(clean_df['node_id'].unique())
    epsilons       = []
    sample_eff     = {w: [] for w in agent_cfg.weights}

    print(f"Training on {len(nodes)} nodes, "
          f"{train_episodes_per_weight} episodes per weight...")

    for ep in range(train_episodes_per_weight):
        for w in agent_cfg.weights:
            agent.ep += 1
            epsilons.append(agent.epsilon())

            # cycle through nodes for diversity
            node = nodes[ep % len(nodes)]
            feed = FITIoTDataFeed(clean_df, node_id=node)
            env  = TQIoTEnv(env_cfg, feed, seed=seed + ep)

            s    = state_to_index(env.reset())
            done = False
            ep_ret = 0.0

            while not done:
                a = agent.select_action(s, w, explore=True)
                obs, (rE, rF), done, _ = env.step(a)
                s_next = state_to_index(obs)
                agent.update(s, a, rE, rF, s_next)
                s = s_next
                ep_ret += w[0] * (-rE) + w[1] * (-rF)

            sample_eff[w].append(ep_ret)

        if (ep + 1) % 25 == 0:
            print(f"  Episode {ep+1}/{train_episodes_per_weight}  "
                  f"epsilon={agent.epsilon():.3f}")

    print("Training complete.")
    return agent, epsilons, sample_eff


agent_cfg = AgentConfig()
env_cfg   = EnvConfig()
agent, epsilons, sample_eff = train(clean_df, agent_cfg, env_cfg,
                                    train_episodes_per_weight=100, seed=42)


## 9. Evaluation — TQ-IoT + Baselines

In [ ]:
def evaluate_policy(agent, w, clean_df, env_cfg, n_episodes=50, seed=999):
    nodes = sorted(clean_df['node_id'].unique())
    all_aoi, all_lt, all_tx = [], [], []

    for ep in range(n_episodes):
        node = nodes[ep % len(nodes)]
        feed = FITIoTDataFeed(clean_df, node_id=node)
        env  = TQIoTEnv(env_cfg, feed, seed=seed + ep)
        s    = state_to_index(env.reset())
        done = False
        aois, tx = [], 0

        while not done:
            a = agent.select_action(s, w, explore=False)
            obs, _, done, _ = env.step(a)
            s = state_to_index(obs)
            aois.append(env.aoi)
            if a == 2: tx += 1

        all_aoi.append(np.mean(aois))
        all_lt.append(len(aois))
        all_tx.append(tx / len(aois) if aois else 0)

    return {
        'mean_aoi': round(np.mean(all_aoi), 3),
        'peak_aoi': round(np.max([max(a) for a in [aois]]), 3),
        'lifetime': round(np.mean(all_lt), 1),
        'tx_rate' : round(np.mean(all_tx), 3),
        'avg_energy': round(np.mean(all_lt) * 10 / max(np.mean(all_lt),1), 3),
    }


def evaluate_baseline(action_id, clean_df, env_cfg, n_episodes=50, seed=999):
    nodes = sorted(clean_df['node_id'].unique())
    all_aoi, all_lt, all_tx = [], [], []

    for ep in range(n_episodes):
        node = nodes[ep % len(nodes)]
        feed = FITIoTDataFeed(clean_df, node_id=node)
        env  = TQIoTEnv(env_cfg, feed, seed=seed + ep)
        env.reset()
        done = False
        aois, tx = [], 0

        while not done:
            _, _, done, _ = env.step(action_id)
            aois.append(env.aoi)
            if action_id == 2: tx += 1

        all_aoi.append(np.mean(aois))
        all_lt.append(len(aois))
        all_tx.append(tx / len(aois) if aois else 0)

    return {
        'mean_aoi': round(np.mean(all_aoi), 3),
        'peak_aoi': round(max([max(a) for a in [aois]]), 3),
        'lifetime': round(np.mean(all_lt), 1),
        'tx_rate' : round(np.mean(all_tx), 3),
        'avg_energy': round(np.mean(all_lt) * {0:1,1:5,2:10}[action_id] / max(np.mean(all_lt),1), 3),
    }


print("Evaluating all 11 TQ-IoT weight configurations...")
rows = []
for wE, wF in agent_cfg.weights:
    metrics = evaluate_policy(agent, (wE, wF), clean_df, env_cfg)
    metrics['policy'] = f'TQ-IoT(wE={wE})'
    metrics['w_E']    = wE
    rows.append(metrics)
    print(f"  wE={wE}  AoI={metrics['mean_aoi']}  "
          f"life={metrics['lifetime']}  tx={metrics['tx_rate']}")

print("\nEvaluating baselines...")
for name, aid in [('Always Sleep',0),('Always Wait',1),('Always Transmit',2)]:
    metrics = evaluate_baseline(aid, clean_df, env_cfg)
    metrics['policy'] = name
    metrics['w_E']    = None
    rows.append(metrics)
    print(f"  {name}: AoI={metrics['mean_aoi']}  "
          f"life={metrics['lifetime']}  tx={metrics['tx_rate']}")

df = pd.DataFrame(rows)
df.to_csv("results/tq_iot_aoi_results.csv", index=False)
print("\nResults saved to results/tq_iot_aoi_results.csv")


## 10. Results Table

In [ ]:
print("\nFULL RESULTS")
print(df[['policy','mean_aoi','lifetime','tx_rate']].to_string(index=False))


## 11. Pareto Front Plot

In [ ]:
learned  = df['w_E'].notna()
baseline = ~learned
tqiot_df = df[learned]
base_df  = df[baseline]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('TQ-IoT on Real FIT IoT-Lab Data — Multi-Objective Results',
             fontsize=13, fontweight='bold')

# ── Pareto front ──────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(tqiot_df['lifetime'], tqiot_df['mean_aoi'],
        'o-', color='#1a5fa8', lw=2, ms=7, label='TQ-IoT policies')

colors = ['#2ca05a','#c0392b','#8e44ad']
markers = ['v','^','D']
for (_, row), col, mk in zip(base_df.iterrows(), colors, markers):
    ax.scatter(row['lifetime'], row['mean_aoi'],
               marker=mk, s=140, color=col, zorder=4, label=row['policy'])

for _, row in tqiot_df.iterrows():
    if row['w_E'] in [0.0, 0.5, 1.0]:
        ax.annotate(f"wE={row['w_E']}", (row['lifetime'], row['mean_aoi']),
                    textcoords='offset points', xytext=(5, 4), fontsize=8)

ax.set_xlabel('Lifetime (steps)')
ax.set_ylabel('Mean AoI')
ax.set_title('Pareto Front')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── AoI vs weight ─────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(tqiot_df['w_E'], tqiot_df['mean_aoi'],
        'o-', color='#1a5fa8', lw=2, ms=6)
for val, col, lb in zip(
    [base_df[base_df['policy']=='Always Sleep']['mean_aoi'].values[0],
     base_df[base_df['policy']=='Always Wait']['mean_aoi'].values[0],
     base_df[base_df['policy']=='Always Transmit']['mean_aoi'].values[0]],
    colors, ['Always Sleep','Always Wait','Always Transmit']):
    ax.axhline(val, color=col, ls=':', lw=1.5, label=lb)
ax.set_xlabel('Energy Weight (wE)')
ax.set_ylabel('Mean AoI')
ax.set_title('AoI vs Energy Weight')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Lifetime vs weight ────────────────────────────────────────────────────────
ax = axes[2]
ax.plot(tqiot_df['w_E'], tqiot_df['lifetime'],
        'o-', color='#e67e22', lw=2, ms=6)
for val, col, lb in zip(
    [base_df[base_df['policy']=='Always Sleep']['lifetime'].values[0],
     base_df[base_df['policy']=='Always Wait']['lifetime'].values[0],
     base_df[base_df['policy']=='Always Transmit']['lifetime'].values[0]],
    colors, ['Always Sleep','Always Wait','Always Transmit']):
    ax.axhline(val, color=col, ls=':', lw=1.5, label=lb)
ax.set_xlabel('Energy Weight (wE)')
ax.set_ylabel('Lifetime (steps)')
ax.set_title('Lifetime vs Energy Weight')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig("results/pareto_front.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: results/pareto_front.png")


## 12. Exploration Decay

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(epsilons)+1), epsilons, lw=2, color='#1a5fa8')
plt.xlabel("Training Episodes")
plt.ylabel("Epsilon (exploration rate)")
plt.title("Exploration Decay over Training")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/epsilon_decay.png", dpi=150, facecolor='white')
plt.show()


## 13. Cumulative Regret

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors_reg = ['#e74c3c','#e67e22','#2ecc71','#3498db','#8e44ad']
labels_reg = ['wE=0.0','wE=0.3','wE=0.5','wE=0.7','wE=1.0']
indices    = [0, 3, 5, 7, 10]

for idx, col, lab in zip(indices, colors_reg, labels_reg):
    w       = agent_cfg.weights[idx]
    returns = np.array(sample_eff[w])
    best    = np.maximum.accumulate(returns)
    regret  = np.cumsum(best - returns)
    ax.plot(range(1, len(regret)+1), regret, lw=2, color=col, label=lab)

ax.set_xlabel("Training Episodes")
ax.set_ylabel("Cumulative Regret")
ax.set_title("Cumulative Regret for Representative Policies")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/cumulative_regret.png", dpi=150, facecolor='white')
plt.show()


## 14. Output Files

| File | Contents |
|---|---|
| `data/processed/fit_iot_clean.csv` | Cleaned sensor data — one row per (node, step) with temp, lux, pressure |
| `results/tq_iot_aoi_results.csv` | All evaluation metrics — 11 TQ-IoT policies + 3 baselines |
| `results/pareto_front.png` | Pareto front + AoI + lifetime plots |
| `results/epsilon_decay.png` | Exploration rate decay over training |
| `results/cumulative_regret.png` | Cumulative regret for 5 representative policies |
